# 03 - Human Approval & Permissions

## Scenario: Northstar Billing Refunds & Database Deletions

Agents must not be granted unchecked administrative access to production systems. If an LLM is compromised or hallucinates, it could delete a database or refund millions of dollars.

In this module, we will explore:
1. **Role-Based Access Control (RBAC)**: Enforcing permissions at the application layer, not just asking the LLM to behave.
2. **The "Prepare & Sign" Pattern**: Using Pydantic to prepare an action for human approval rather than executing it.
3. **Cryptographic Signing (Mock)**: Ensuring the payload wasn't tampered with.

In [1]:
from pydantic import BaseModel
from typing import Optional
import json

# Define the Agent's identity
AGENT_ROLE = "tier_1_support"

# Define our mock Database
def delete_customer_record(customer_id: str, role: str):
    """Simulates a database deletion with RBAC enforcement."""
    if role != "admin":
        raise PermissionError(f"Role '{role}' is not authorized to execute DELETIONS.")
    print(f"✅ Customer {customer_id} DELETED from database.")


## 1. Application-Layer RBAC

Do NOT rely on the System Prompt for security. (e.g., "You are not allowed to delete users"). The LLM can easily be tricked into ignoring that rule. The security MUST live in your Python code.

In [2]:
def tool_delete_customer(customer_id: str):
    print(f"Agent attempting to delete {customer_id}...")
    try:
        # We pass the agent's hardcoded role to the DB layer
        delete_customer_record(customer_id, role=AGENT_ROLE)
    except PermissionError as e:
        print(f"🚨 ACTION BLOCKED: {e}")
        return f"Error: {e}"

# Watch the agent get blocked
tool_delete_customer("cust_992")


Agent attempting to delete cust_992...
🚨 ACTION BLOCKED: Role 'tier_1_support' is not authorized to execute DELETIONS.


"Error: Role 'tier_1_support' is not authorized to execute DELETIONS."

## 2. The Prepare & Sign Pattern (Human-in-the-loop)

If the agent isn't allowed to issue refunds, what *can* it do? It can prepare a **Proposal**. 
We force the LLM to output a Pydantic schema representing the proposed action, which we then send to a human for approval.

In [3]:
class RefundProposal(BaseModel):
    customer_id: str
    amount: float
    justification: str

def prepare_refund(customer_id: str, amount: float, justification: str) -> str:
    # 1. Validate the proposal using Pydantic
    try:
        proposal = RefundProposal(
            customer_id=customer_id, 
            amount=amount, 
            justification=justification
        )
    except ValueError as e:
        return f"Validation Error: {e}"
        
    # 2. Serialize and "Store" for human review
    payload = proposal.model_dump_json()
    
    # In a real system, you would insert this into a DB and trigger a Slack message to an Admin
    print("\n🔔 [SLACK ALERT to Admins]")
    print(f"Agent has requested a refund. Please review payload:")
    print(payload)
    print("-------------------------------------------------")
    
    # 3. Tell the LLM to stop and wait
    return "Refund proposal submitted to Admins. Inform the user we are waiting for approval."

print(prepare_refund("cust_123", 49.99, "SLA violation on checkout service."))



🔔 [SLACK ALERT to Admins]
Agent has requested a refund. Please review payload:
{"customer_id":"cust_123","amount":49.99,"justification":"SLA violation on checkout service."}
-------------------------------------------------
Refund proposal submitted to Admins. Inform the user we are waiting for approval.


## Checkpoint

**1. Where is the correct place to enforce permissions for an Agent?**
- A) In the System Prompt (e.g., "Do not delete databases").
- B) In the Application/API layer using standard RBAC, checking the Agent's identity before executing the tool.
- C) By asking the user for a password before running the tool.
- D) In the vector database.

**2. What is the benefit of the "Prepare & Sign" pattern?**
- A) It makes the LLM run faster.
- B) It allows the LLM to execute dangerous tools securely.
- C) It restricts the LLM to merely generating structured data (Proposals) which a human can safely review and execute later.
- D) It encrypts the LLM's memory.
